# Support Integrity Auditor (SIA) - Full Reproducible Pipeline

This notebook contains the complete reproducible pipeline for the Support Integrity Auditor (SIA). It includes:
1. **Stage 1: Pseudo-Label Generation (Self-Supervised)**
2. **Stage 2: Classifier Training (Fine-Tuning on CPU)**
3. **Stage 3: Evidence Dossier Generation and Evaluation**

---

## Setup & Imports

In [ ]:
import os
import re
import time
import json
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, f1_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

## Stage 1: Pseudo-Label Generation (Self-Supervised)
We fuse three signals:
- **Semantic Urgency Score** via `all-MiniLM-L6-v2` cosine similarity differences.
- **Rule-based NLP Urgency Score** via keyphrase/negation density.
- **Resolution-Time Regression Score** via Ridge regression prediction of resolution complexity.

In [ ]:
df = pd.read_csv('enhanced_customer_support_data.csv')
df['text'] = df['Ticket_Subject'].fillna('') + " " + df['Ticket_Description'].fillna('')

print("1. Computing Embedding-based Semantic Urgency...")
st_model = SentenceTransformer('all-MiniLM-L6-v2')

high_anchors = [
    "system down crash database error outage locked",
    "cannot access account login failed credentials hacked",
    "security breach data leak credit card fraud identity theft",
    "payment failure charge failed unable to purchase service blocked",
    "critical error crash loop failure broken"
]
low_anchors = [
    "general inquiry question question hours of operation location",
    "how do i update profile settings email configuration info",
    "pricing plans enterprise packages information request",
    "feature request suggestion dark mode enhancement feedback",
    "hello team appreciation thank you feedback greetings"
]

high_embeds = st_model.encode(high_anchors)
low_embeds = st_model.encode(low_anchors)
embeddings = st_model.encode(df['text'].tolist(), show_progress_bar=True)

sim_high = cosine_similarity(embeddings, high_embeds).max(axis=1)
sim_low = cosine_similarity(embeddings, low_embeds).max(axis=1)
sem_score_raw = sim_high - sim_low
scaler = MinMaxScaler()
df['sem_score'] = scaler.fit_transform(sem_score_raw.reshape(-1, 1))

print("2. Computing Rule-based NLP Urgency...")
urgent_words = [
    r"urgent", r"emergency", r"immediate", r"asap", r"critical", r"broken", r"down",
    r"crash", r"hacked", r"stolen", r"breach", r"fail", r"error", r"block", r"locked",
    r"cannot access", r"can't log", r"unable to", r"not working", r"not syncing",
    r"failed to", r"security", r"preventing", r"stop", r"freeze", r"frozen", r"leak"
]
escalation_words = [
    r"manager", r"supervisor", r"escalate", r"escalation", r"refund", r"cancel", 
    r"cancellation", r"chargeback", r"legal", r"lawyer", r"sue", r"court", r"complaint", 
    r"worst", r"terrible", r"disappointed"
]

def get_rule_score(text):
    text_lower = text.lower()
    score = 0
    for w in urgent_words:
        if re.search(w, text_lower):
            score += 1
    for w in escalation_words:
        if re.search(w, text_lower):
            score += 2
    return score

df['rule_raw_score'] = df['text'].apply(get_rule_score)
df['rule_score'] = scaler.fit_transform(df[['rule_raw_score']])

print("3. Computing Resolution-time Regression Score...")
ridge = Ridge(alpha=1.0)
ridge.fit(embeddings, df['Resolution_Time_Hours'])
predicted_res_time = ridge.predict(embeddings)
df['res_score'] = scaler.fit_transform(predicted_res_time.reshape(-1, 1))

print("4. Fusing Scores and Mapping Severity...")
df['fused_score'] = 0.5 * df['sem_score'] + 0.3 * df['rule_score'] + 0.2 * df['res_score']

fused_vals = df['fused_score'].values
p_low_val = np.percentile(fused_vals, 38.6)
p_med_val = np.percentile(fused_vals, 76.4)
p_high_val = np.percentile(fused_vals, 93.5)

def map_severity(score):
    if score <= p_low_val:
        return 'Low'
    elif score <= p_med_val:
        return 'Medium'
    elif score <= p_high_val:
        return 'High'
    else:
        return 'Critical'

df['inferred_severity'] = df['fused_score'].apply(map_severity)

level_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
df['assigned_num'] = df['Priority_Level'].map(level_map)
df['inferred_num'] = df['inferred_severity'].map(level_map)
df['mismatch'] = (np.abs(df['inferred_num'] - df['assigned_num']) >= 2).astype(int)

print(df['mismatch'].value_counts())
df.to_csv('pseudo_labeled_data.csv', index=False)

## Stage 2: Classifier Training (Fine-Tuning BERT-Tiny)
We load Google's `bert_uncased_L-2_H-128_A-2` and fine-tune it for 2 epochs on the generated pseudo-labels.

In [ ]:
df['input_text'] = (
    "Priority: " + df['Priority_Level'] + 
    " | Category: " + df['Issue_Category'] + 
    " | Channel: " + df['Ticket_Channel'] + 
    " | Subject: " + df['Ticket_Subject'] + 
    " | Description: " + df['Ticket_Description']
)
dataset_df = df[['input_text', 'mismatch']].rename(columns={'mismatch': 'label'})

train_df = dataset_df.sample(frac=0.8, random_state=42)
val_df = dataset_df.drop(train_df.index)

model_name = "google/bert_uncased_L-2_H-128_A-2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def tokenize_function(examples):
    return tokenizer(examples['input_text'], truncation=True, max_length=128, padding='max_length')

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

class_weights = len(train_df) / (2.0 * np.bincount(train_df['label']))
print("Class weights:", class_weights)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=model.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    rec_0 = recall_score(labels, predictions, pos_label=0)
    rec_1 = recall_score(labels, predictions, pos_label=1)
    return {
        'accuracy': acc,
        'macro_f1': f1,
        'recall_consistent': rec_0,
        'recall_mismatch': rec_1
    }

training_args = TrainingArguments(
    output_dir='./results_notebook',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    use_cpu=True
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()
print(trainer.evaluate())